# Lab 6 — Multi-Source Retrieval with Foundry IQ

**Optional / preview-dependent · 60 minutes · Level 200**

Connect a Foundry prompt agent to a prepared Foundry IQ knowledge base through
its MCP endpoint. The knowledge base can plan subqueries across multiple
knowledge sources and return source references.

## Learning objectives

- Understand knowledge bases, knowledge sources, and agentic retrieval.
- Attach the `knowledge_base_retrieve` MCP tool to a prompt agent.
- Use an explicit grounded-answer contract.
- Delete only the namespaced agent version created by this notebook.

Foundry IQ availability depends on region and API version. Some capabilities
in the `2026-05-01-preview` API remain preview and carry no production SLA.

In [ ]:
import os
import re

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)


def safe_name(value: str, *, max_length: int = 40) -> str:
    value = re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
    value = re.sub(r"-+", "-", value)
    if not value:
        raise ValueError("Resource namespace must contain a letter or number.")
    return value[:max_length].rstrip("-")


raw_namespace = (
    os.getenv("WORKSHOP_RESOURCE_NAMESPACE")
    or os.getenv("WORKSHOP_TEAM_ID")
    or os.getenv("WORKSHOP_PARTICIPANT_ID")
)
if not raw_namespace:
    raise ValueError(
        "Set WORKSHOP_RESOURCE_NAMESPACE (preferred), WORKSHOP_TEAM_ID, "
        "or WORKSHOP_PARTICIPANT_ID before running workshop labs."
    )

RESOURCE_NAMESPACE = safe_name(raw_namespace)
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL = os.environ["FOUNDRY_MODEL"]

print(f"Namespace: {RESOURCE_NAMESPACE}")
print(f"Model: {MODEL}")

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import InteractiveBrowserCredential

SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
KNOWLEDGE_BASE_NAME = os.getenv(
    "FOUNDRY_IQ_KNOWLEDGE_BASE",
    f"{RESOURCE_NAMESPACE}-grid-operations-kb",
)
MCP_CONNECTION_NAME = os.getenv(
    "FOUNDRY_IQ_MCP_CONNECTION_NAME",
    f"{RESOURCE_NAMESPACE}-grid-operations-kb-connection",
)
IQ_API_VERSION = os.getenv(
    "FOUNDRY_IQ_API_VERSION", "2026-05-01-preview"
)
AGENT_NAME = safe_name(
    f"iq-operations-{RESOURCE_NAMESPACE}", max_length=48
)
MCP_ENDPOINT = (
    f"{SEARCH_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp"
    f"?api-version={IQ_API_VERSION}"
)

print(f"Knowledge base: {KNOWLEDGE_BASE_NAME}")
print(f"MCP connection: {MCP_CONNECTION_NAME}")
print(f"Agent version owner: {RESOURCE_NAMESPACE}")


## Participant task

Update `QUESTION` so it genuinely requires information from more than one
source—for example, a procedure plus an ownership or evidence requirement.

In [ ]:
credential = InteractiveBrowserCredential()
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
mcp_tool = MCPTool(
    server_label="grid-operations-knowledge",
    server_url=MCP_ENDPOINT,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=MCP_CONNECTION_NAME,
)
instructions = '''
Use the knowledge base for every factual answer.
Never answer an operational fact from model memory.
If the knowledge base does not contain enough evidence, respond with
"I don't know based on the available sources."
Include source annotations in every grounded answer.
Keep operational recommendations advisory and require human approval.
'''

created_agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=instructions,
        tools=[mcp_tool],
    ),
    description="Namespaced workshop agent for multi-source grid knowledge.",
)
print(
    f"Created {created_agent.name} version {created_agent.version}"
)


In [ ]:
# TODO(participant): adapt this grounded three-source question.
QUESTION = '''
For a synthetic incident in postcode 1704 requiring VWI E-85, identify an
available crew member and their valid raamopdracht that covers E-85. Also
summarize the documented E-85 procedure requirements. Separate procedure,
authorization, and crew evidence.
'''

openai_client = project.get_openai_client()
conversation = openai_client.conversations.create()
response = openai_client.responses.create(
    conversation=conversation.id,
    input=QUESTION,
    extra_body={
        "agent_reference": {
            "name": created_agent.name,
            "type": "agent_reference",
        }
    },
)
print(response.output_text)

## Deterministic success check

In [ ]:
assert response.output_text.strip(), "Foundry IQ agent returned no text."
assert created_agent.name == AGENT_NAME
assert RESOURCE_NAMESPACE in created_agent.name
assert "knowledge_base_retrieve" in mcp_tool.allowed_tools
assert "E-85" in response.output_text
assert "source" in response.output_text.lower()
print("PASS — the namespaced agent returned a knowledge-base response.")

## Safe cleanup

Delete only the conversation and agent version created in this run. The shared
MCP connection, knowledge base, knowledge sources, and indexes are intentionally
preserved.

In [ ]:
if (
    created_agent.name == AGENT_NAME
    and RESOURCE_NAMESPACE in created_agent.name
):
    try:
        openai_client.conversations.delete(conversation_id=conversation.id)
        print(f"Deleted owned IQ conversation: {conversation.id}")
    finally:
        project.agents.delete_version(
            agent_name=created_agent.name,
            agent_version=created_agent.version,
        )
        print(
            f"Deleted owned agent version: "
            f"{created_agent.name}/{created_agent.version}"
        )
else:
    raise RuntimeError("Cleanup refused: agent is not owned by this namespace.")

## Fallback

If Foundry IQ is unavailable, complete Lab 5 and discuss how application-owned
query decomposition could call multiple direct Search indexes. Do not silently
replace Foundry IQ with an unrelated feature.

## Optional extension

Compare minimal, low, and medium retrieval reasoning effort in a facilitator-
managed environment and record quality, latency, and token trade-offs.

**Expected artifact:** a cited multi-source answer and safe agent-version cleanup.